# Keshen endometrium microarray ML pipeline — leak-free nested CV

In [1]:
# ==============================================================================
# STEP 1: IMPORTS & DATA LOADING
# ==============================================================================
print("🕒 Step 1: Imports & data loading…")

import pandas as pd
import numpy as np
from joblib import dump
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, GroupShuffleSplit,
    GridSearchCV, cross_validate
)
from sklearn.ensemble import RandomForestClassifier

# Load & align data
try:
    expr = pd.read_csv("data/GENE_MATRIX_COMBO_MAR31.csv", index_col=0).T
    meta = pd.read_csv("data/METADATA_COMBO_MAR31_covarsRemoved.csv", index_col=0)
except FileNotFoundError:
    print("\n⚠️ Data files not found. Please ensure 'data/GENE_MATRIX_COMBO_MAR31.csv' and")
    print("   'data/METADATA_COMBO_MAR31_covarsRemoved.csv' are in the correct directory.")



expr = expr.loc[meta.index] # Align expression data rows to metadata rows
y = (meta["condition"] == "RIF").astype(int).values
groups = meta["study"].values # Groups for study-aware splitting

print(f"   samples={expr.shape[0]}, genes={expr.shape[1]}, classes={np.bincount(y)}")
print("✅ Step 1 complete.")

🕒 Step 1: Imports & data loading…
   samples=217, genes=10489, classes=[121  96]
✅ Step 1 complete.


In [2]:
# ==============================================================================
# STEP 2: DEFINE LEAK-SAFE COMBO SELECTOR
# ==============================================================================
print("\n🕒 Step 2: Defining ComboSelector (ANOVA + MI)…")

class ComboSelector(BaseEstimator, TransformerMixin):
    """Inside-CV: sum ANOVA+MI, keep top-k."""
    def __init__(self, k=30):
        self.k = k
    def fit(self, X, y):
        # Ensure X is numpy array for sklearn functions
        X_ = X.values if hasattr(X, "values") else X
        if X_.shape[1] == 0: # Handle case where no features remain
             self.mask_ = np.zeros(X_.shape[1], bool)
             self.scores_ = np.array([])
             return self
        # Calculate scores only if there are features
        f_vals, _ = f_classif(X_, y)
        # Handle potential NaNs from f_classif (e.g., zero variance features)
        f_vals = np.nan_to_num(f_vals)
        mi_vals = mutual_info_classif(X_, y, random_state=0)

        combined_scores = f_vals + mi_vals
        self.scores_ = combined_scores # Store scores for potential inspection

        # Determine number of features to keep (cannot exceed available features)
        k_actual = min(self.k, X_.shape[1])
        if k_actual > 0:
            idx = np.argsort(combined_scores)[-k_actual:]
            self.mask_ = np.zeros(X_.shape[1], dtype=bool)
            self.mask_[idx] = True
        else:
             self.mask_ = np.zeros(X_.shape[1], dtype=bool) # No features to select
        return self

    def transform(self, X):
        X_ = X.values if hasattr(X, "values") else X
        # Check if mask was created (i.e., fit was called)
        if not hasattr(self, "mask_"):
             raise RuntimeError("Transformer has not been fitted yet.")
        # Check if any features were selected
        if not np.any(self.mask_):
             # Return an array with the correct number of samples but zero features
             return np.empty((X_.shape[0], 0))
        return X_[:, self.mask_]

    def get_support(self, indices=False):
        """Get a mask, or integer indices, of the features selected."""
        if indices:
            return np.where(self.mask_)[0]
        return self.mask_

    # Add get_feature_names_out for compatibility with recent sklearn versions
    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            # Fallback if input_features not provided (less ideal)
             if hasattr(self, 'mask_'):
                 return np.array([f"feature_{i}" for i in np.where(self.mask_)[0]])
             else:
                 return np.array([]) # Or raise error
        input_features = np.asarray(input_features)
        if hasattr(self, 'mask_'):
             return input_features[self.mask_]
        else:
            # Handle case before fit or if fit resulted in no mask
            return np.array([])


print("✅ Step 2 complete.")


🕒 Step 2: Defining ComboSelector (ANOVA + MI)…
✅ Step 2 complete.


In [3]:
# ==============================================================================
# STEP 3: DEFINE CV SPLITTERS, PIPELINE & GRID SEARCH
# ==============================================================================
print("\n🕒 Step 3: Defining CV splitters, pipeline, and hyperparameter grid…")

# 3.1 – Define INNER cross-validator (for hyperparameter tuning)
#       Needs to be defined *before* GridSearchCV
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
print(f"   Inner CV splitter: {inner_cv}")

# 3.2 – Define OUTER cross-validator (for pipeline evaluation)
#       Using GroupShuffleSplit to prevent leakage between studies
#       Using n_splits=10 as mentioned in the initial description text.
outer_cv = GroupShuffleSplit(n_splits=10, test_size=0.2, random_state=1)
print(f"   Outer CV splitter: {outer_cv}")

# 3.3 – Build the preprocessing step
#       Uses ComboSelector (k tuned by grid search) and StandardScaler
#       Applied via ColumnTransformer to handle pandas DataFrame input
numeric_features = expr.columns # Get all gene names
numeric_transformer = Pipeline(steps=[
    ('select', ComboSelector(k=30)),  # Default k, will be tuned by GridSearchCV
    ('scale', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('genes', numeric_transformer, numeric_features)
    ],
    remainder='drop' # Drop any columns not specified (shouldn't be any here)
)
print("   Preprocessor defined (ColumnTransformer with ComboSelector + StandardScaler)")

# 3.4 – Assemble the full pipeline
#       Combines preprocessing and the RandomForest classifier
pipeline = Pipeline(steps=[
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=1  # Let GridSearchCV/cross_validate handle parallelism for folds/jobs
    ))
])
print("   Full pipeline assembled (Preprocessor -> RandomForest)")

# 3.5 – Define the hyperparameter grid for GridSearchCV
#       Tunes 'k' for ComboSelector and 'max_depth' for RandomForest
param_grid = {
    'prep__genes__select__k': [30, 60, 120], # Referencing step name 'select' inside 'genes' transformer inside 'prep' step
    'clf__max_depth': [None, 20]           # Referencing 'max_depth' in 'clf' step
}
print(f"   Grid-search parameters:")
print(f"     k (prep__genes__select__k) ∈ {param_grid['prep__genes__select__k']}")
print(f"     max_depth (clf__max_depth) ∈ {param_grid['clf__max_depth']}")

# 3.6 – Create the GridSearchCV object for the inner loop
#       This object encapsulates the pipeline and the hyperparameter search logic
#       It uses the 'inner_cv' defined earlier.
grid_search_cv = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=inner_cv,          # Use the inner StratifiedKFold splitter
    scoring='roc_auc',    # Metric for evaluating hyperparameter combinations
    n_jobs=16,            # Parallelize inner CV folds & param combinations if backend supports it
    verbose=0             # Set to 1 or higher for more messages during grid search
)
print(f"   GridSearchCV object created, using inner_cv for tuning.")
print("✅ Step 3 complete — ready for nested CV in Step 4.")


🕒 Step 3: Defining CV splitters, pipeline, and hyperparameter grid…
   Inner CV splitter: StratifiedKFold(n_splits=5, random_state=1, shuffle=True)
   Outer CV splitter: GroupShuffleSplit(n_splits=10, random_state=1, test_size=0.2, train_size=None)
   Preprocessor defined (ColumnTransformer with ComboSelector + StandardScaler)
   Full pipeline assembled (Preprocessor -> RandomForest)
   Grid-search parameters:
     k (prep__genes__select__k) ∈ [30, 60, 120]
     max_depth (clf__max_depth) ∈ [None, 20]
   GridSearchCV object created, using inner_cv for tuning.
✅ Step 3 complete — ready for nested CV in Step 4.


In [4]:
# ==============================================================================
# STEP 4: RUN NESTED CROSS-VALIDATION
# ==============================================================================
print("\n🕒 Step 4: Running nested CV (outer loop evaluation)…")

# Use cross_validate to run the outer loop.
# It takes the GridSearchCV object (grid_search_cv) as the estimator.
# For each split generated by outer_cv:
#   1. grid_search_cv.fit(X_outer_train, y_outer_train) is called.
#      This performs the inner cross-validation (using inner_cv) on the outer training data
#      to find the best hyperparameters (k, max_depth) for that specific outer fold.
#   2. The best estimator found by GridSearchCV (pipeline refit with best params on X_outer_train)
#      is used to predict/score on the outer test set (X_outer_test, y_outer_test).
# The 'groups' parameter ensures GroupShuffleSplit uses the study information correctly.
# n_jobs=-1 parallelizes the outer folds.
cv_results = cross_validate(
    estimator=grid_search_cv, # The GridSearchCV object IS the estimator here
    X=expr,
    y=y,
    groups=groups,            # Crucial for GroupShuffleSplit
    cv=outer_cv,              # The study-aware outer splitter
    scoring='roc_auc',        # Final evaluation metric on the outer test folds
    return_estimator=True,    # Return the fitted GridSearchCV object for each outer fold
    n_jobs=-1,                # Use all available CPU cores for the outer loop
    verbose=1                 # Show progress for outer folds
)

# Calculate and report performance
mean_auc = np.mean(cv_results['test_score'])
std_auc = np.std(cv_results['test_score'])
print(f"\n✅ Step 4 complete.")
print(f"   Leak-free Nested CV AUROC = {mean_auc:.3f} ± {std_auc:.3f}")
print(f"   Individual outer fold scores: {[f'{s:.3f}' for s in cv_results['test_score']]}")




🕒 Step 4: Running nested CV (outer loop evaluation)…


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}

In [ ]:
# ==============================================================================
# STEP 5: ANALYZE RESULTS - SELECTED GENES & BEST MODEL
# ==============================================================================
print("\n🕒 Step 5: Analyzing selected features and saving the 'best' outer fold model…")

# Check how many genes were selected in each outer fold by the best inner model for that fold
selected_k_per_fold = []
best_inner_scores_per_outer_fold = []

# The estimators returned by cross_validate are the fitted GridSearchCV objects
# from each outer fold.
for idx, estimator_gscv in enumerate(cv_results['estimator']):
    # estimator_gscv is a fitted GridSearchCV instance for this outer fold
    best_pipeline_for_fold = estimator_gscv.best_estimator_
    selector_step = best_pipeline_for_fold.named_steps['prep'].named_transformers_['genes'].named_steps['select']

    # Get the number of features selected by the best estimator for this fold
    n_selected = selector_step.get_support().sum()
    selected_k_per_fold.append(n_selected)

    # Store the best score achieved during the inner CV grid search for this outer fold
    best_inner_scores_per_outer_fold.append(estimator_gscv.best_score_)

    # Optional: Print details for each outer fold
    # print(f"   Outer Fold {idx+1}: Best Inner Score={estimator_gscv.best_score_:.4f}, Selected k={n_selected}, Best Params={estimator_gscv.best_params_}")


print(f"   Genes kept (k) per outer fold (best inner model): {selected_k_per_fold}")
print(f"   Best inner CV scores per outer fold: {[f'{s:.3f}' for s in best_inner_scores_per_outer_fold]}")

# --- Choosing and Saving the "Best" Model ---
# Note: In nested CV, the main output is the performance estimate (mean_auc ± std_auc).
# There isn't one single "best" model trained during nested CV itself.
# Common approaches for a final model:
#   1. Retrain GridSearchCV on ALL data to find the overall best hyperparameters,
#      then train a final pipeline using these params on ALL data. (Recommended for deployment)
#   2. Select the GridSearchCV object from the outer fold that achieved the highest
#      *inner* cross-validation score (estimator_gscv.best_score_). This was the approach
#      in the original notebook. We'll follow this for consistency.

best_outer_fold_index = np.argmax(best_inner_scores_per_outer_fold)
best_gscv_estimator = cv_results['estimator'][best_outer_fold_index]
# The final model to save is the *best pipeline* found by this chosen GridSearchCV object
final_model_pipeline = best_gscv_estimator.best_estimator_

print(f"\n   Selected outer fold {best_outer_fold_index + 1} as 'best' based on highest inner CV score ({best_inner_scores_per_outer_fold[best_outer_fold_index]:.4f}).")
print(f"   Best hyperparameters found in this fold: {best_gscv_estimator.best_params_}")

# Save the best pipeline found within that best GridSearchCV object
model_filename = "rf_keshen_leakfree_pipeline.joblib"
dump(final_model_pipeline, model_filename)
print(f"✅ Step 5 complete. 'Best' pipeline saved to {model_filename}")

In [ ]:
# ==============================================================================
# STEP 6: REPORT OVERALL BEST HYPERPARAMETERS (from the chosen 'best' fold)
# ==============================================================================
print("\n🕒 Step 6: Reporting best hyperparameters from the chosen 'best' outer fold…")
# We already extracted this in Step 5, just printing again clearly.
best_params = best_gscv_estimator.best_params_
print(f"   Best hyperparameters (from outer fold {best_outer_fold_index + 1} with highest inner CV score):")
print(f"     prep__genes__select__k: {best_params.get('prep__genes__select__k', 'N/A')}")
print(f"     clf__max_depth: {best_params.get('clf__max_depth', 'N/A')}")
print("✅ Step 6 complete.")

In [ ]:
# ==============================================================================
# STEP 7: VISUALIZATION — AUROC DISTRIBUTION & TOP FEATURES
# ==============================================================================
print("\n🕒 Step 7: Plotting AUROC distribution & feature importances from the saved model…")

# 1) AUROC histogram from outer folds
plt.figure(figsize=(7, 5))
plt.hist(cv_results["test_score"], bins=min(len(cv_results["test_score"]), 10), edgecolor="k", alpha=0.7)
plt.title(f"Nested CV Outer Fold AUROC Distribution (n={len(cv_results['test_score'])})")
plt.xlabel("AUROC Score (Outer Test Fold)")
plt.ylabel("Frequency (Number of Outer Folds)")
plt.axvline(mean_auc, color="red", linestyle="--", label=f"Mean AUROC: {mean_auc:.3f}")
plt.axvline(mean_auc + std_auc, color="grey", linestyle=":", label=f"+1 Std Dev: {(mean_auc + std_auc):.3f}")
plt.axvline(mean_auc - std_auc, color="grey", linestyle=":", label=f"-1 Std Dev: {(mean_auc - std_auc):.3f}")
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# 2) Top-10 feature importances from the SAVED model (final_model_pipeline)
#    Need to access the steps within the saved pipeline
try:
    # Access the fitted RandomForestClassifier step
    rf_classifier = final_model_pipeline.named_steps['clf']

    # Access the fitted ComboSelector step to get the final selected feature mask/names
    # Navigate through ColumnTransformer -> Pipeline -> ComboSelector
    preprocessor_step = final_model_pipeline.named_steps['prep']
    numeric_pipeline = preprocessor_step.named_transformers_['genes']
    selector_step = numeric_pipeline.named_steps['select']

    # Get the boolean mask of selected features from the selector
    selected_mask = selector_step.get_support()

    # Get the names of the features that were originally fed into the selector
    # These should correspond to the columns used in the ColumnTransformer
    original_feature_names = preprocessor_step.transformers_[0][2] # ('genes', numeric_transformer, numeric_features)

    # Get the names of the *selected* features using the mask
    selected_feature_names = np.array(original_feature_names)[selected_mask]

    # Ensure the number of importances matches the number of selected features
    if len(rf_classifier.feature_importances_) == len(selected_feature_names):
        importances = rf_classifier.feature_importances_

        # Get indices of top 10 importances
        top_indices = np.argsort(importances)[-10:] # Indices of the smallest to largest

        plt.figure(figsize=(8, 6))
        plt.barh(selected_feature_names[top_indices], importances[top_indices], color='skyblue')
        plt.xlabel("Feature Importance (Gini Impurity Reduction)")
        plt.ylabel("Gene")
        plt.title("Top 10 Feature Importances from Saved Model")
        plt.tight_layout()
        plt.show()
    else:
        print("\n⚠️ Warning: Mismatch between number of importances and selected features.")
        print(f"   Importances array length: {len(rf_classifier.feature_importances_)}")
        print(f"   Selected feature names count: {len(selected_feature_names)}")

except AttributeError as e:
    print(f"\n⚠️ Error accessing parts of the saved pipeline for feature importance plot: {e}")
except Exception as e:
    print(f"\n⚠️ An unexpected error occurred during plotting: {e}")


print("✅ Step 7 complete. Analysis finished.")